# Build Climatologies from 5 Years of ROMS Output

## Load Modules

In [1]:
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, zoomed_inset_axes
import matplotlib.ticker as mticker
from matplotlib import path
from matplotlib.colors import TwoSlopeNorm
import matplotlib.cm as cm
from matplotlib.patches import Polygon
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors

import xarray as xr
from netCDF4 import Dataset
from xgcm import Grid
import cmaps
import cmocean
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import glob

import rioxarray

from dask.diagnostics import ProgressBar

from scipy.ndimage import gaussian_filter1d

from pyproj import Transformer

import seawater as sw
import gsw

import calendar

import pickle

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.06/lib/python3.12/site-packages/sitecustomize.py:72: UserWarning: The seawater library is deprecated! Please use gsw instead.
  mod = _real_import(name, globals, locals, fromlist, level)


## Define Resuable Functions

In [2]:
def inpolygon(xq, yq, xv, yv):
    shape = xq.shape
    xq = xq.reshape(-1)
    yq = yq.reshape(-1)
    xv = xv.reshape(-1)
    yv = yv.reshape(-1)
    q = [(xq[i], yq[i]) for i in range(xq.shape[0])]
    p = path.Path([(xv[i], yv[i]) for i in range(xv.shape[0])])
    return p.contains_points(q).reshape(shape)

def ROMSmetricsAndGrid(ds, grd):
    ds = ds.rename({'eta_u': 'eta_rho', 'xi_v': 'xi_rho', 'xi_psi': 'xi_u', 'eta_psi': 'eta_v'})

    coords={'X':{'center':'xi_rho', 'inner':'xi_u'}, 
        'Y':{'center':'eta_rho', 'inner':'eta_v'}, 
        'Z':{'center':'s_rho', 'outer':'s_w'}}

    grid = Grid(ds, coords=coords, periodic=[], autoparse_metadata=False)

    print('making pm/pn metrics')
    ds['pm_v'] = grid.interp(ds.pm, 'Y')
    ds['pn_u'] = grid.interp(ds.pn, 'X')
    ds['pm_u'] = grid.interp(ds.pm, 'X')
    ds['pn_v'] = grid.interp(ds.pn, 'Y')
    ds['pm_psi'] = grid.interp(grid.interp(ds.pm, 'Y'),  'X') # at psi points (eta_v, xi_u) 
    ds['pn_psi'] = grid.interp(grid.interp(ds.pn, 'X'),  'Y') # at psi points (eta_v, xi_u)

    print('making dx/dy')
    ds['dx'] = 1/ds.pm
    ds['dx_u'] = 1/ds.pm_u
    ds['dx_v'] = 1/ds.pm_v
    ds['dx_psi'] = 1/ds.pm_psi

    ds['dy'] = 1/ds.pn
    ds['dy_u'] = 1/ds.pn_u
    ds['dy_v'] = 1/ds.pn_v
    ds['dy_psi'] = 1/ds.pn_psi

#     ds['dz'] = grid.diff(ds.z_w, 'Z', boundary='fill')
#     ds['dz_w'] = grid.diff(ds.z_rho, 'Z', boundary='fill')
#     ds['dz_u'] = grid.interp(ds.dz, 'X')
#     ds['dz_w_u'] = grid.interp(ds.dz_w, 'X')
#     ds['dz_v'] = grid.interp(ds.dz, 'Y')
#     ds['dz_w_v'] = grid.interp(ds.dz_w, 'Y')

    ds['dA'] = ds.dx * ds.dy

    metrics = {
        ('X',): ['dx', 'dx_u', 'dx_v', 'dx_psi'], # X distances
        ('Y',): ['dy', 'dy_u', 'dy_v', 'dy_psi'], # Y distances
        # ('Z',): ['dz', 'dz_u', 'dz_v', 'dz_w', 'dz_w_u', 'dz_w_v'], # Z distances
        ('X', 'Y'): ['dA'] # Areas
    }
    grid = Grid(ds, coords=coords, metrics=metrics, periodic=[], autoparse_metadata=False)

    ds = ds.assign_coords({"x_rho": grd.x_rho})
    ds = ds.assign_coords({"y_rho": grd.y_rho})
    ds = ds.assign_coords({"lon_rho": grd.lon_rho})
    ds = ds.assign_coords({"lat_rho": grd.lat_rho})
    
    # ds['hc'] = grd.hc
    # ds['s_w']=grd.s_w
    # ds['s_rho']=grd.s_rho
    # ds['Cs_w']=grd.Cs_w
    # ds['Cs_r']=grd.Cs_r
    
    # ds_raw = ds_raw.assign_coords(lon_rho=grd.lon_rho)
    # ds_raw = ds_raw.assign_coords(lat_rho=grd.lat_rho)
    
    createMasks(ds)
    
    weights_area = (1/ds.pm)*(1/ds.pn)
    weights_area.name = "weights"

    print('making vertical coordinates')
    Zo_rho = (ds.hc * ds.s_rho + ds.Cs_r * ds.h) / (ds.hc + ds.h)
    z_rho = (ds.zice) + ((ds.zice) + ds.h) * Zo_rho
    Zo_w = (ds.hc * ds.s_w + ds.Cs_w * ds.h) / (ds.hc + ds.h)
    z_w = Zo_w * ((ds.zice) + ds.h) + (ds.zice)

    ds.coords['z_w0'] = z_w.where(ds.mask_rho, 0).transpose('s_w', 'eta_rho', 'xi_rho')
    ds.coords['z_rho0'] = z_rho.where(ds.mask_rho, 0).transpose('s_rho', 'eta_rho', 'xi_rho')

    ds['dz'] = (('s_rho', 'eta_rho', 'xi_rho'),np.diff(ds.z_w0,axis=0))

    print('mapping u/v to u/v rho')
    ds['u_rho'] = grid.interp(ds.u,'X')
    ds['v_rho'] = grid.interp(ds.v,'Y')
    ds['ubar_rho'] = grid.interp(ds.ubar, 'X')
    ds['vbar_rho'] = grid.interp(ds.vbar, 'Y')

    return ds,grid

def add_zeros_to_4(date):
    if date<10:
        to_add = '000'
    elif date>9 & date<100:
        to_add = '00'
    elif date>99 & date < 1000:
        to_add = '0'
    else: 
        to_add = ''
    return to_add

def generateFileList(FilePath,prefix,datelist):
    filelist=[FilePath+prefix+add_zeros_to_4(datelist[0])+str(datelist[0])+'.nc']
    for dates in datelist[1:]:
        filenameToAppend=FilePath+prefix+add_zeros_to_4(dates)+str(dates)+'.nc'
        filelist.append(filenameToAppend)
    return filelist

def createMasks(ds):
    ds['mask_zice'] = (ds.mask_rho.where(ds.zice<0)==1) # make a logic mask for ice
    ds['mask_openocean'] = ds.mask_rho*(ds.mask_zice==0)
    ds['mask_openoceanNaN'] = ds.mask_openocean.where(ds.mask_openocean)
    ds['mask_rhoNaN'] = ds.mask_rho.where(ds.mask_rho)
    ds['mask_ziceNaN'] = ds.mask_zice.where(ds.mask_zice)

    poly_shelf = np.array([
        [1.55e6,600000],
        [2.25e6,600000],
        [2.25e6,800000],
        [2.20e6,830000],
        [1.55e6,890000]])
    
    mask_roi = inpolygon(ds.x_rho.values, ds.y_rho.values,poly_shelf[:,0], poly_shelf[:,1])
    ds['mask_zice_roi'] = ds.mask_zice*mask_roi
    roi_label1 = 'amery'
    ds.mask_zice_roi.attrs['long_name']=roi_label1

## Load Datasets

In [3]:
# ---------- GRID ----------
grd = xr.open_dataset('/g/data/jk72/deg581/apom/frc/amery_v2km_v13_grd.nc')

# ---------- MODEL OUTPUT ----------
FilePath='/g/data/gv90/bkg581/prd_amery/amery_CTRL/' # Truth file settings
prefix='roms_avg_'
timeRange = [49, 54]
datelist = np.array(range(timeRange[0],timeRange[1],1)) 

fl=generateFileList(FilePath,prefix,datelist)

ds = xr.open_mfdataset(fl,chunks = {'ocean_time':1}, data_vars='minimal', compat='override',coords='minimal',parallel='False',join='right')

## Handle Loaded Datasets

In [4]:
# ---------- MODEL OUTPUT ----------
ds = ds.drop_vars(['sustr','svstr','ssflux','shflux','AKt','AKv','Hsbl','w'])
ds['my'] = ds.m*(60*60*24*365)

ds, grid = ROMSmetricsAndGrid(ds, grd)

ds_CTRL = ds
ds.close()

# ---------- GRID ----------
grd['hc']=ds_CTRL.hc
grd['s_w']=ds_CTRL.s_w
grd['s_rho']=ds_CTRL.s_rho
grd['Cs_w']=ds_CTRL.Cs_w
grd['Cs_r']=ds_CTRL.Cs_r

createMasks(grd)

making pm/pn metrics
making dx/dy
making vertical coordinates
mapping u/v to u/v rho


## Construct Climatologies

In [5]:
No_of_years=5
dayofyear = np.concatenate([np.array((356,361)),np.tile(np.arange(1,365,5),No_of_years-1),np.arange(1,365-10,5)])
dayofyear_da = xr.DataArray(dayofyear,[('day',dayofyear)])

# Subset levels and rechunk time coordinates before building climatologies
ds_CTRL['day']=dayofyear_da

# CALCULATE CLIMATOLOGIES
# Raw pot temp/prac salt
salt_raw_doy = ds_CTRL.salt.assign_coords(dayofyear=("ocean_time", dayofyear_da.data)).groupby('dayofyear').mean()
temp_raw_doy = ds_CTRL.temp.assign_coords(dayofyear=("ocean_time", dayofyear_da.data)).groupby('dayofyear').mean()

# Convert raw temp/salt climatologies via gsw to conservative temp and absolute salinity
kw = dict(dask='parallelized', output_dtypes=[float])
ds_CTRL['pressure'] = xr.apply_ufunc(gsw.p_from_z, ds_CTRL.z_rho0, ds_CTRL.lat_rho, **kw)
ds_CTRL['salt_doy'] = xr.apply_ufunc(gsw.SA_from_SP, salt_raw_doy, ds_CTRL.pressure, ds_CTRL.lon_rho, ds_CTRL.lat_rho, **kw)
ds_CTRL['temp_doy'] = xr.apply_ufunc(gsw.CT_from_pt, ds_CTRL.salt_doy, temp_raw_doy, **kw)

ds_CTRL['my_doy']=ds_CTRL.my.assign_coords(dayofyear=("ocean_time",dayofyear_da.data)).groupby('dayofyear').mean()
ds_CTRL['u_rho_doy']=ds_CTRL.u_rho.assign_coords(dayofyear=("ocean_time",dayofyear_da.data)).groupby('dayofyear').mean()
ds_CTRL['v_rho_doy']=ds_CTRL.v_rho.assign_coords(dayofyear=("ocean_time",dayofyear_da.data)).groupby('dayofyear').mean()
ds_CTRL['ubar_rho_doy'] = ds_CTRL.ubar_rho.assign_coords(dayofyear=("ocean_time", dayofyear_da.data)).groupby('dayofyear').mean()
ds_CTRL['vbar_rho_doy'] = ds_CTRL.vbar_rho.assign_coords(dayofyear=("ocean_time", dayofyear_da.data)).groupby('dayofyear').mean()

## Remove Obsolete Variables and Load the Model Output

In [6]:
# ds_CTRL = ds_CTRL.drop_vars(['u_rho_doy','v_rho_doy','temp_doy','u_rho','v_rho'])
ds_CTRL = ds_CTRL.drop_vars(['m','Tstar','Ustar','u','v','temp','salt','rho','zeta','Tb','my', 'u_rho','v_rho', 'ubar', 'vbar'])
print(ds_CTRL.nbytes/1e9,'G')

40.109673352 G


In [7]:
with ProgressBar():
    ds_CTRL = ds_CTRL.load()

[########################################] | 100% Completed | 19m 50s


In [ ]:
# SAVE TO NETCDF, including compression
# (ONLY DO THIS ONCE)
encoding = {v: {'zlib': True, 'complevel': 4} for v in ds_CTRL.data_vars}
print('encoding done')
ds_CTRL.to_netcdf('/g/data/gv90/ad6790/amery-amisor/data/proc/ctrl_climatology.nc', encoding=encoding)
print('save complete')

# delayed = ds_CTRL.to_netcdf('/g/data/gv90/ad6790/amery-amisor/data/proc/ctrl_climatology.nc', compute=False)
# with ProgressBar():
#     delayed.compute()

encoding done
